# Homework 3: Shadow analysis

September 15, 2025 · 08:00–17:00 EDT · Philadelphia.

Adapted from [Xiaojiang Li’s PyCUDA notebook](https://colab.research.google.com/drive/1V5FUkRBuGOnoUH1q9hNHiUzg_4XzMU5F). [DSM GeoTIFF](https://drive.google.com/file/d/1narrEJ4lVbFDHGAVx1HLJNh7uWZcrlVr/view).

In Colab, select **Runtime → Change runtime type → T4 GPU**. Save the DSM in your Google Drive and update `dsm_file` below to its location. Run all cells.

DSM elevations are assumed to be US survey feet; the file does not specify vertical units. Both elevations and pixel spacing are converted to meters. Shadows from objects outside the DSM are not included.

## Prepare the modules

In [ ]:
!pip install pycuda
!pip install rasterio pillow

In [ ]:
from pycuda.compiler import SourceModule
import pycuda.autoinit
import pycuda.driver as cuda
import os, os.path
import rasterio as rio
import numpy as np
from google.colab import drive
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Image as DisplayImage

%matplotlib inline
from pylab import rcParams
rcParams['figure.figsize'] = (8, 8)

print("%d device(s) found." % cuda.Device.count())
for ordinal in range(cuda.Device.count()):
    dev = cuda.Device(ordinal)
    print("Device #%d: %s" % (ordinal, dev.name()))

## Load the DSM

In [ ]:
drive.mount('/content/drive')

dsm_file = "/content/drive/MyDrive/Colab Notebooks/data/row11-col17.tif"
dsm_dataset = rio.open(dsm_file)
dsm_bounds = dsm_dataset.bounds
dsm_img = dsm_dataset.read(1).astype(np.float32)
print('The dsm bound is:', dsm_bounds)

transform = dsm_dataset.transform
pixel_width = transform[0]
pixel_height = -transform[4]

feet_to_meters = 1200 / 3937
dsm_img = (dsm_img * feet_to_meters).astype(np.float32)
cell_size = pixel_width * feet_to_meters
dsm_dataset.close()

output_dir = "/content/drive/MyDrive/Colab Notebooks/Homework3-output/2025-09-15"
os.makedirs(output_dir, exist_ok=True)

## Solar positions

Precalculated with pvlib’s NREL SPA method for September 15, 2025, at the DSM center (40.021297° N, 75.082970° W), using America/New_York time. Each tuple is `(hour, elevation, azimuth)` in degrees; azimuth is clockwise from true north. The 1.730° grid convergence correction aligns the sun direction with this DSM’s projected grid. These values can be checked with the [NOAA calculator](https://gml.noaa.gov/grad/solcalc/azel.html).

In [ ]:
solar_pos_lst = [
    (8, 14.195321, 98.353621),
    (9, 25.308174, 109.023104),
    (10, 35.668461, 121.613324),
    (11, 44.523796, 137.478388),
    (12, 50.687021, 157.835286),
    (13, 52.711194, 181.903891),
    (14, 49.940780, 205.583261),
    (15, 43.244499, 225.186036),
    (16, 34.074810, 240.421644),
    (17, 23.542751, 252.610290),
]
grid_convergence = 1.730

## Calculate the shadows

In [ ]:
kernel_shadow = """
#define PI 3.1415926

__global__ void calculate_shadow(float *dsm, bool *shadow, int width, int height, float cell_size, float sun_azimuth, float sun_elevation) {
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;

    if (x >= width || y >= height)
        return;

    int idx = y * width + x;

    float azimuth_rad = sun_azimuth * 3.14159 / 180.0;
    float elevation_rad = sun_elevation * 3.14159 / 180.0;

    float dx = sin(azimuth_rad);
    float dy = -cos(azimuth_rad);
    float dz = tan(elevation_rad);

    float initial_height = dsm[idx];
    bool in_shadow = false;

    for (float t = cell_size; t < 2000.0f; t += cell_size) {
        int x_offset = x + int(dx * t / cell_size);
        int y_offset = y + int(dy * t / cell_size);

        if (x_offset < 0 || x_offset >= width || y_offset < 0 || y_offset >= height)
            break;

        int offset_idx = y_offset * width + x_offset;

        float height_along_ray = initial_height + dz * t;

        if (dsm[offset_idx] > height_along_ray) {
            in_shadow = true;
            break;
        }
    }

    shadow[idx] = in_shadow;
}
"""

mod = SourceModule(kernel_shadow)
calculate_shadows = mod.get_function("calculate_shadow")

In [ ]:
height, width = dsm_img.shape
shadow_result = np.zeros_like(dsm_img, dtype=bool)

d_dsm = cuda.mem_alloc(dsm_img.nbytes)
d_shadow = cuda.mem_alloc(shadow_result.nbytes)
cuda.memcpy_htod(d_dsm, dsm_img)

block_size = (16, 16, 1)
grid_size = (int(np.ceil(width / block_size[0])), int(np.ceil(height / block_size[1])))

## Export hourly PNGs

In [ ]:
png_files = []

for hour, sunele, azimuth in solar_pos_lst:
    sun_azimuth = azimuth - grid_convergence
    sun_elevation = sunele

    calculate_shadows(
        d_dsm, d_shadow, np.int32(width), np.int32(height),
        np.float32(cell_size), np.float32(sun_azimuth), np.float32(sun_elevation),
        block=block_size, grid=grid_size
    )
    cuda.memcpy_dtoh(shadow_result, d_shadow)
    binary_shadow = shadow_result.astype(np.uint8)

    plt.figure()
    plt.imshow(binary_shadow, cmap="gray_r", vmin=0, vmax=1)
    plt.colorbar(ticks=[0, 1], label="0 = sunlit, 1 = shadow")
    plt.title("Shadow Distribution: September 15, 2025, %02d:00 EDT" % hour)
    plt.axis("off")
    png_file = os.path.join(output_dir, "%s.png" % hour)
    plt.savefig(png_file, dpi=150)
    plt.show()
    plt.close()
    png_files.append(png_file)

In [ ]:
d_dsm.free()
d_shadow.free()

## Combine PNGs into a GIF

In [ ]:
frames = []
for png_file in png_files:
    with Image.open(png_file) as image:
        frames.append(image.convert("RGB"))

gif_file = os.path.join(output_dir, "shadows_08-17.gif")
frames[0].save(gif_file, save_all=True, append_images=frames[1:], duration=800, loop=0, disposal=2)
display(DisplayImage(filename=gif_file))
print("PNG files and GIF saved to:", output_dir)